# kvcalc on a T4: predicted vs reported KV cache, and what caps concurrency

GPU session required (Kaggle T4). ~10 min install, ~10 min for two server launches, ~10 min sweep.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install recipe from inference-infra-portfolio/KAGGLE.md (each line earned by a failed servellm run).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/kvcalc" "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, kvcalc.calc, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("imports ok, vllm", vllm.__version__)

## 1. Predict, then launch, at two context lengths

The prediction uses the driver-reported memory and the same utilisation vLLM will use.

In [ ]:
import json, subprocess, torch
from kvcalc.calc import ModelSpec, capacity
from kvcalc.cli import _load_spec
from servellm.launch import ServerConfig, VllmServer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"driver-reported VRAM: {vram_gib:.2f} GiB")
import httpx
cfg = httpx.get(f"https://huggingface.co/{MODEL}/resolve/main/config.json", follow_redirects=True).json()
spec = ModelSpec.from_config(cfg, name=MODEL, n_params=1.54e9)

rows = []
for max_len in (2048, 8192):
    pred = capacity(spec, vram_gib=vram_gib, utilization=0.9, max_model_len=max_len)
    server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=max_len, gpu_memory_utilization=0.9),
                        log_path=f"results/startup_{max_len}.log")
    info = server.start(ready_timeout_s=900)
    server.stop()
    rows.append({"max_model_len": max_len, "predicted_tokens": pred["kv_tokens"], "predicted_kv_gib": pred["kv_budget_gib"],
                 "reported_tokens": info.kv_cache_tokens, "reported_kv_gib": info.kv_cache_gib,
                 "reported_max_concurrency": info.max_concurrency, "predicted_max_concurrency": pred["max_concurrency_at_max_len"],
                 "model_load_gib": info.model_load_gib, "attention_backend": info.attention_backend})
    print(rows[-1])
json.dump({"vram_gib": vram_gib, "rows": rows}, open("results/prediction_vs_reported.json", "w"), indent=1)

In [ ]:
for r in rows:
    if r["reported_tokens"]:
        err = (r["predicted_tokens"] - r["reported_tokens"]) / r["reported_tokens"] * 100
        print(f"max_model_len={r['max_model_len']}: predicted {r['predicted_tokens']:,} vs reported {r['reported_tokens']:,} tokens ({err:+.1f}%)")
    else:
        print("startup log wording changed; see results/startup_*.log")

## 2. Monitor during a load sweep

Server at 4096 max length; monitor samples once a second while tokenbench climbs to 64 concurrent requests. Watch whether `waiting` rises while KV usage is still well below 100% (then the cap is `max-num-seqs`) or only once usage nears 100% (memory).

In [ ]:
import subprocess, time
server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9), log_path="results/startup_4096.log")
info = server.start(ready_timeout_s=900)
pred = capacity(spec, vram_gib=vram_gib, utilization=0.9, max_model_len=4096)
mon = subprocess.Popen(["kvcalc", "monitor", "--base-url", "http://127.0.0.1:8000", "--kv-tokens", str(info.kv_cache_tokens or pred["kv_tokens"]),
                        "--interval", "1", "--duration", "420", "--out", "results/monitor_timeline.json"], stderr=open("results/monitor.log", "w"))
!tokenbench sweep --base-url http://127.0.0.1:8000 --model {MODEL} --out results/sweep.json --levels 1,4,16,64,128 --num-requests 64 --prompt-tokens 1024 --max-tokens 256 --plots
mon.wait(timeout=600)
server.stop()
print(open("results/monitor.log").read()[-3000:])

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip